# M1a-FULL — full-scale MSD base + pull-push term (Colab, resume-safe)

Runs **M1a-FULL**: the original MSD recipe from the upstream appendix, plus the *same*
pull-push term M1a used. Everything writes **straight to Drive**, and the run is
**resume-safe**: if the Colab runtime drops, just re-run this notebook top-to-bottom — it
picks up from the last completed epoch. **You never lose more than one epoch of work.**

**Recipe (hard-locked, from `locuslab/robust_union` `CIFAR10/train.py` + `cifar_funcs.py`)**

| | value | source |
|---|---|---|
| epochs | **50** | their `train.py` |
| batch / wd / momentum | **128 / 5e-4 / 0.9** | their `train.py` |
| lr | **one-cycle, peak 0.1** — `np.interp(t, [0,20,40,50], [0,0.1,0.005,0])` | their `train.py` |
| MSD attack | **`msd_v0`, 50 iters**, alphas (0.003, 0.05, 0.05) | their `cifar_funcs.py` |
| threat triple | **ℓ∞ 8/255 · ℓ2 0.5 · ℓ1 12** | **ours** (deliberate deviation — see below) |
| term (== M1a) | 3 APGD views @ 10 steps, α=β=0.5, τ=0.1, warm-up 10 ep, head 512→512→128 | imported verbatim from the M1a trainer |
| seed | **0** | |

> **One threat triple everywhere: ℓ∞ 8/255, ℓ2 0.5, ℓ1 12.** Upstream trains at ℓ∞
> **0.03**; we deliberately do not. Everything else in the base recipe is theirs — only the
> radius changes. Training, the val-select proxy, and the audit therefore all use the *same*
> triple, so M1a-FULL is directly comparable to M1a / M0 (which also trained at 8/255) and
> there is no train/audit threat-model gap to caveat. The triple is **imported** from the
> M1a trainer rather than restated, so it cannot silently drift from the other arms.

> **Why not `torch.OneCycleLR`:** their schedule is a plain `np.interp` of the epoch, so it
> is a pure function of the epoch counter — nothing to checkpoint, and resume restores it
> exactly. That is a feature, not a shortcut.

## Cell 1 — Setup: GPU · Drive · code

**Pick a GPU first:** Runtime → Change runtime type → GPU. **Prefer A100** if Colab offers
it; the MSD attack at 50 iters is ~5× the cost of the 10-iter version, so the GPU tier
dominates the wall-clock (see the ETA the training cell prints).

Code comes from **`attackdro_code.zip` on Drive**, the same pattern the audit notebooks
use — the GitHub repo is private, so Colab cannot clone it anonymously, and making it
public during anonymous review would be a bad trade. Re-upload the zip whenever the
trainer changes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv 2>/dev/null || echo 'NO GPU -- Runtime > Change runtime type > GPU'

from google.colab import drive
drive.mount('/content/drive')

import zipfile, os
DRIVE    = '/content/drive/MyDrive/attackdro'
CODE_ZIP = f'{DRIVE}/attackdro_code.zip'
REPO     = '/content/attackdro'

assert os.path.exists(CODE_ZIP), (
    f'{CODE_ZIP} not found on Drive. Upload the rebuilt attackdro_code.zip '
    '(it must contain scripts/dev/train_full_msd.py).')
os.makedirs(REPO, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(REPO)
    names = z.namelist()

need = ['scripts/dev/train_full_msd.py', 'scripts/dev/c5_fromscratch.py',
        'src/robustdro/attacks/norms.py', 'src/robustdro/attacks/apgd_train.py',
        'src/robustdro/models/preact_resnet.py']
missing = [f for f in need if f not in names]
assert not missing, (f'code zip is STALE -- missing {missing}. '
                     'Rebuild and re-upload attackdro_code.zip.')

import torch
print('code    :', REPO, f'({len(names)} files from the zip)')
print('torch   :', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Cell 1b — W&B login (track the run live)

Logs to project **`attackdro-union`**, run **`C5_M1a_full_seed0`**. The run id is
deterministic and the logger uses `resume="allow"`, so every Colab re-attach **continues the
same run** — one continuous curve, not a new run per session.

**Set the Colab secret once** (sidebar 🔑 → add `WANDB_API_KEY`, value from
<https://wandb.ai/authorize>). Then every future re-attach logs in automatically, with no
prompt — which matters, because a ~13 h run will re-attach several times. Without W&B the
training still runs and still prints to stdout; it just isn't tracked.

In [ ]:
import subprocess, importlib.util
if importlib.util.find_spec('wandb') is None:
    subprocess.run(['pip', 'install', '-q', 'wandb'])
import wandb

key = None
try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')          # sidebar key icon -> add this secret
except Exception:
    pass

if key:
    wandb.login(key=key)
    print('wandb: logged in from the Colab secret -- re-attaches will be automatic.')
else:
    print('No WANDB_API_KEY Colab secret found; falling back to an interactive login.')
    print('Strongly recommended: sidebar key icon -> add secret WANDB_API_KEY')
    print('(value from https://wandb.ai/authorize). Otherwise you must paste the key')
    print('again after EVERY Colab disconnect, and this run will see several.')
    wandb.login()

print('\nproject: attackdro-union | run: C5_M1a_full_seed0 (resumes in place)')

## Cell 2 — Config (Drive-direct: every write lands on Drive)

In [ ]:
DRIVE  = '/content/drive/MyDrive/attackdro'
OUTDIR = f'{DRIVE}/C5_full/M1a_full'          # ckpt_latest.pt, val_best.pt, train.json all land HERE
CIFAR_TARGZ = f'{DRIVE}/cifar-10-python.tar.gz'
SEED = 0
import os; os.makedirs(OUTDIR, exist_ok=True)
print('OUTDIR (Drive-direct):', OUTDIR)
print('  -> ckpt_latest.pt  written after EVERY epoch (resume point)')
print('  -> val_best.pt     best val worst-union so far')
print('  -> train.json      full history, rewritten every epoch; epochs_completed==50 = done-sentinel')

## Cell 3 — CIFAR-10 (reuse the uploaded tarball; hash-checked)

In [ ]:
import hashlib, tarfile, os
os.makedirs(f'{REPO}/data', exist_ok=True)
if not os.path.exists(f'{REPO}/data/cifar-10-batches-py/train_batch') and \
   not os.path.exists(f'{REPO}/data/cifar-10-batches-py/data_batch_1'):
    if os.path.exists(CIFAR_TARGZ):
        h = hashlib.sha256(open(CIFAR_TARGZ, 'rb').read()).hexdigest()
        assert h.startswith('6d958be074577803'), f'CIFAR tarball sha mismatch: {h[:16]}'
        with tarfile.open(CIFAR_TARGZ) as t: t.extractall(f'{REPO}/data')
        print('extracted the uploaded CIFAR-10 tarball (hash verified)')
    else:
        import torchvision
        torchvision.datasets.CIFAR10(f'{REPO}/data', train=True, download=True)
        torchvision.datasets.CIFAR10(f'{REPO}/data', train=False, download=True)
        print('downloaded CIFAR-10 (no tarball on Drive)')
assert os.path.exists(f'{REPO}/data/cifar-10-batches-py/data_batch_1'), 'CIFAR-10 missing'
print('CIFAR-10 ready at', f'{REPO}/data')

## Cell 4 — Resume status (read-only; just tells you where you are)

In [ ]:
import json, os, torch
latest = f'{OUTDIR}/ckpt_latest.pt'
tj     = f'{OUTDIR}/train.json'
if os.path.exists(latest):
    st = torch.load(latest, map_location='cpu', weights_only=False)
    done = int(st['epoch']) + 1
    print(f'RESUME: {done}/50 epochs already done (best valWU so far {float(st["best"]):.4f}).')
    print(f'        the training cell will continue from epoch {done} -- nothing to do.')
elif os.path.exists(tj):
    print('train.json exists but ckpt_latest.pt does not -- fresh start (history will be overwritten).')
else:
    print('FRESH START: no ckpt_latest.pt on Drive; training begins at epoch 0.')

## Cell 5 — Train (streams live; prints pace + ETA after 2 epochs)

Safe to interrupt or lose the runtime at any point: re-run the notebook and it resumes from
`ckpt_latest.pt` on Drive. Re-running when already finished is a no-op.

In [ ]:
import subprocess, re, os, sys

cmd = [sys.executable, f'{REPO}/scripts/dev/train_full_msd.py',
       '--outdir', OUTDIR, '--seed', str(SEED), '--wandb-mode', 'online']
env = dict(os.environ, ATTACKDRO_ROOT=REPO, C5_NUM_WORKERS='2')
print('$', ' '.join(cmd), '\n')

EP = re.compile(r'ep(\d+)/(\d+).*?\((\d+)s\)')
secs, warned = [], False
proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
    m = EP.search(line)
    if m:
        done, total, sec = int(m.group(1)), int(m.group(2)), int(m.group(3))
        secs.append(sec)
        if len(secs) == 2 or (len(secs) > 2 and done % 10 == 0):
            pace = sum(secs[-2:]) / 2
            rem_h = (total - done) * pace / 3600
            print(f'\n>>> pace {pace:.0f}s/epoch | {total - done} epochs left '
                  f'| ETA {rem_h:.1f}h (finishes ~{rem_h:.1f}h from now)\n')
            if rem_h > 24 and not warned:
                warned = True
                print('>>> WARNING: ETA exceeds 24h. This is FINE -- the run is resume-safe.\n'
                      '>>> Colab will drop the runtime before then; just re-open this notebook\n'
                      '>>> and run all cells again. It continues from the last finished epoch.\n'
                      '>>> Expect to re-attach the session a few times. Consider an A100.\n')
rc = proc.wait()
print(f'\n[exit {rc}]')

## Cell 6 — Done? → hand off to the audit

The done-sentinel is `train.json` reaching `epochs_completed == 50`.

In [ ]:
import json, os
tj = f'{OUTDIR}/train.json'
JOB = [
    "     {'name':'M1a_full',",
    "      'ckpt': f'{DRIVE}/C5_full/M1a_full/val_best.pt',",
    "      'family':'robustdro', 'scales':['10k'], 'enabled':True, 'gate':'done',",
    "      'note':'full-scale MSD base + pull-push term'},",
]
if not os.path.exists(tj):
    print('train.json not on Drive yet -- training has not completed an epoch.')
else:
    d = json.load(open(tj))
    done = d.get('epochs_completed', 0)
    print(f'epochs_completed: {done}/50 | best val worst-union (proxy): {d.get("best_val_worst_union"):.4f}')
    if done < 50:
        print(f'\nNOT finished. Re-run Cell 5 to continue from epoch {done}.')
    else:
        print('\nDONE. The proxy number above is NOT the reported result -- audit next:\n')
        print('  1. Open audit_gate_colab.ipynb (GPU runtime).')
        print('  2. Add this job to its QUEUE:')
        for ln in JOB:
            print(ln)
        print('  3. Result lands at union_bench/M1a_full/10k/{eval.json, masks_multinorm_v1.npz}.')
        print('\n  NOTE: that notebook_s done-gate requires epoch >= 70, which fits the 80-epoch')
        print('  arms. This run is 50 epochs BY DESIGN, so the gate would wrongly skip it.')
        print('  Pass min_epoch=45 for this job, or drop the gate key (train.json already')
        print('  proves completion). Do not raise the epoch count to satisfy the gate.')